# Week 7 — Day 6: Gradio App — Intel Image Classifier

## Goal
Build a deployed image classifier app with Gradio.
Users can upload any image and get predictions with confidence scores.

---

## What is Gradio?
Gradio is a Python library that turns any ML model into a web app
in under 10 lines of code. No web development knowledge needed.
- Input: image upload box
- Output: bar chart of class probabilities
- share=True: generates public URL anyone can access

---

## App Architecture

```python
def predict(image):
    # 1. convert numpy array to PIL Image
    image = Image.fromarray(image).convert('RGB')
    
    # 2. apply same transforms as training
    tensor = transform(image).unsqueeze(0).to(device)
    
    # 3. forward pass
    with torch.no_grad():
        output = model_app(tensor)
        probs  = torch.softmax(output, dim=1)[0]
    
    # 4. return dict — Gradio reads this as bar chart
    return {classes[i]: float(probs[i]) for i in range(len(classes))}

app = gr.Interface(
    fn=predict,
    inputs=gr.Image(),
    outputs=gr.Label(num_top_classes=6),
    title="Intel Image Classifier",
    description="Upload an image — model predicts: buildings, forest,
                 glacier, mountain, sea, or street"
)

app.launch(share=True)
```

---

## Key Concepts

### predict() function flow


### Why softmax here?
Model outputs raw scores (logits) — can be any value.
softmax converts to probabilities that sum to 1.0.
Makes output interpretable — 98% glacier is meaningful.

### Why transform must match training?
Model learned from 224×224 ImageNet-normalized images.
Different preprocessing = different input distribution = wrong predictions.
Always use identical transform pipeline for inference.

---

## Model — Common Issue Fixed

### Problem — Random predictions (16.86% per class)
Model saved before fine-tuning completed.
state_dict saved correctly but weights were from frozen-only training.

### Fix
Retrain completely and save as new file immediately after training:
```python
torch.save(model.state_dict(),
           '/content/drive/MyDrive/ml-journey/intel_efficientnet_v2.pth')
```

### Verification step — always do this after loading
```python
for idx in range(5):
    image, label = test_dataset[idx]
    tensor = image.unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(tensor)
        pred   = output.argmax(dim=1).item()
    print(f"True: {classes[label]} | Pred: {classes[pred]}")
```
If all predictions are same class with ~16% confidence → model broken.
If predictions vary with high confidence → model working correctly.

---

## Final Model Performance
- Frozen training (5 epochs): 89.47%
- Fine-tuning (5 epochs): 93.67%
- Glacier image test: 98% confidence ✅

---

## Deployment

### Gradio Public URL
```python
app.launch(share=True)
# generates: https://xxxxxxxx.gradio.live
# valid for 72 hours
```

### Hugging Face Spaces
Gradio SDK now requires payment on Hugging Face.
Free alternatives:
- Streamlit SDK on Hugging Face (free)
- Render.com (free permanent hosting)
- GitHub Pages for static demos

### Files needed for deployment


---

## Colab Session Management — Lessons Learned

Always save to Google Drive, not /content/:
```python
# wrong — deleted when session resets
torch.save(model.state_dict(), 'model.pth')

# correct — permanent
torch.save(model.state_dict(),
           '/content/drive/MyDrive/ml-journey/model.pth')
```

Always save dataset to Drive after first download:
```python
os.system('cp -r /content/intel /content/drive/MyDrive/ml-journey/intel_dataset')
```

Load from Drive next session:
```python
os.system('cp -r /content/drive/MyDrive/ml-journey/intel_dataset /content/intel')
```

---

## Key Takeaways
- Gradio turns any predict() function into a web app in 10 lines
- predict() receives numpy array — convert to PIL first
- Transform pipeline must be identical to training
- softmax converts raw scores to probabilities for display
- Always verify model predictions before deploying
- save to /content/drive/MyDrive/ not /content/ — permanent storage
- Model weights + app.py + requirements.txt = complete deployment package
- Gradio share=True link valid 72 hours — screenshot for portfolio
- Next: Day 7 — GitHub update + internship applications + Week 8 prep

In [ ]:
import os

# setup kaggle
os.makedirs('/root/.kaggle', exist_ok=True)
os.system('cp kaggle.json /root/.kaggle/')
os.system('chmod 600 /root/.kaggle/kaggle.json')

# download fresh
os.system('kaggle datasets download -d puneet6060/intel-image-classification -p /content/')
print(os.listdir('/content'))

['.config', 'drive', 'intel', 'archive (2).zip', '.gradio', 'intel-image-classification.zip', 'sample_data']


In [ ]:
import os
os.system('unzip /content/intel-image-classification.zip -d /content/intel')
print(os.listdir('/content/intel'))

['seg_train', 'seg_test', 'seg_pred']


In [ ]:
os.system('cp -r /content/intel /content/drive/MyDrive/ml-journey/intel_dataset')
print("Saved to Drive!")

Saved to Drive!


# Retrain The Model :

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms
from torch.utils.data import DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# transforms
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# dataloaders
train_dataset = IntelDataset('/content/intel/seg_train/seg_train', transform=train_transform)
test_dataset  = IntelDataset('/content/intel/seg_test/seg_test',  transform=val_transform)
train_loader  = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=2)
test_loader   = DataLoader(test_dataset,  batch_size=32, shuffle=False, num_workers=2)
print(f"Train: {len(train_dataset)} | Test: {len(test_dataset)}")

# model
model = models.efficientnet_b0(weights='IMAGENET1K_V1')
for param in model.parameters():
    param.requires_grad = False
model.classifier[1] = nn.Linear(1280, 6)
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=0.001)

# train frozen 5 epochs
print("\n--- Frozen Training ---")
for epoch in range(5):
    model.train()
    correct = 0; total = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        output = model(images)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()
        correct += (output.argmax(1) == labels).sum().item()
        total   += labels.size(0)
    model.eval()
    vc = 0; vt = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            vc += (model(images).argmax(1) == labels).sum().item()
            vt += labels.size(0)
    print(f"Epoch {epoch+1} | Train: {correct/total:.4f} | Test: {vc/vt:.4f}")

# fine-tune 5 epochs
print("\n--- Fine-tuning ---")
for param in model.parameters():
    param.requires_grad = True
optimizer = optim.Adam(model.parameters(), lr=1e-4)

for epoch in range(5):
    model.train()
    correct = 0; total = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        output = model(images)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()
        correct += (output.argmax(1) == labels).sum().item()
        total   += labels.size(0)
    model.eval()
    vc = 0; vt = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            vc += (model(images).argmax(1) == labels).sum().item()
            vt += labels.size(0)
    print(f"Epoch {epoch+1} | Train: {correct/total:.4f} | Test: {vc/vt:.4f}")

# save immediately
torch.save(model.state_dict(), '/content/drive/MyDrive/ml-journey/intel_efficientnet_v2.pth')
print("\nModel saved to Drive!")

Train: 14034 | Test: 3000

--- Frozen Training ---
Epoch 1 | Train: 0.8272 | Test: 0.8857
Epoch 2 | Train: 0.8710 | Test: 0.8857
Epoch 3 | Train: 0.8796 | Test: 0.8873
Epoch 4 | Train: 0.8821 | Test: 0.8903
Epoch 5 | Train: 0.8826 | Test: 0.8947

--- Fine-tuning ---
Epoch 1 | Train: 0.9109 | Test: 0.9263
Epoch 2 | Train: 0.9478 | Test: 0.9343
Epoch 3 | Train: 0.9651 | Test: 0.9340
Epoch 4 | Train: 0.9753 | Test: 0.9367
Epoch 5 | Train: 0.9814 | Test: 0.9293

Model saved to Drive!


# Verify it works:

In [ ]:
model2 = models.efficientnet_b0(weights=None)
model2.classifier[1] = nn.Linear(1280, 6)
model2.load_state_dict(torch.load('/content/drive/MyDrive/ml-journey/intel_efficientnet_v2.pth',
                                   map_location=device))
model2 = model2.to(device)
model2.eval()

classes = ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']

for idx in range(5):
    image, label = test_dataset[idx]
    tensor = image.unsqueeze(0).to(device)
    with torch.no_grad():
        output = model2(tensor)
        probs  = torch.softmax(output, dim=1)[0]
        pred   = output.argmax(dim=1).item()
    print(f"True: {classes[label]:12s} | Pred: {classes[pred]:12s} | Conf: {probs[pred]:.2%}")

True: buildings    | Pred: street       | Conf: 99.18%
True: buildings    | Pred: buildings    | Conf: 99.80%
True: buildings    | Pred: buildings    | Conf: 96.34%
True: buildings    | Pred: buildings    | Conf: 99.87%
True: buildings    | Pred: buildings    | Conf: 77.85%


# Install Gradio:

In [ ]:
!pip install gradio
import gradio as gr
print(gr.__version__)

6.26.0


# Build the Gradio app locally first:

In [ ]:
import gradio as gr
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
classes = ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']

# load correct model
model_app = models.efficientnet_b0(weights=None)
model_app.classifier[1] = nn.Linear(1280, 6)
model_app.load_state_dict(torch.load('/content/drive/MyDrive/ml-journey/intel_efficientnet_v2.pth',
                                      map_location=device))
model_app = model_app.to(device)
model_app.eval()

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

def predict(image):
    image = Image.fromarray(image).convert('RGB')
    tensor = transform(image).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model_app(tensor)
        probs  = torch.softmax(output, dim=1)[0]
    return {classes[i]: float(probs[i]) for i in range(len(classes))}

app = gr.Interface(
    fn=predict,
    inputs=gr.Image(),
    outputs=gr.Label(num_top_classes=6),
    title="Intel Image Classifier",
    description="Upload an image — model predicts: buildings, forest, glacier, mountain, sea, or street"
)

app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0a29e339a9025204f7.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
from huggingface_hub import notebook_login
notebook_login()